[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/02_tres_pilares/02_tres_pilares.ipynb)

# Módulo 02 · Los tres pilares: traces, metrics y logs (y el modelo de datos de OpenTelemetry)

| | |
|---|---|
| **Nivel** | 🟢 Básico |
| **Duración estimada** | ≈ 4 h 30 min (lección ≈ 2 h 45 min + proyecto ≈ 1 h 45 min) |
| **Servicios de Compose** | **Ninguno** (usa tráfico simulado y los datos pregrabados del repo). Funciona igual en Colab |
| **RAM aproximada** | < 400 MB (solo el kernel de Python) |
| **Prerrequisitos** | Módulo 00 (mapa del stack, OTLP/JSON a mano) y módulo 01 (SupportOps v0, fallos inyectados, `RecordingInstrumentor`) |
| **Hilo conductor** | Aprendes a *leer* telemetría de SupportOps antes de producirla con el SDK (módulo 03) |

## Objetivos de aprendizaje

Al terminar esta lección podrás:

1. **Explicar** qué pregunta responde cada señal (logs, métricas, trazas) y por qué un agente necesita las tres.
2. **Construir a mano** un contador, un *gauge* y un histograma, y **estimar percentiles** a partir de sus *buckets* (con su error).
3. **Describir** el modelo de datos de OpenTelemetry: *resource*, *instrumentation scope*, *span* (kind, status, atributos, eventos, enlaces), *log record* y tipos de métrica con su temporalidad.
4. **Propagar contexto** con la cabecera W3C `traceparent` (implementarla a mano) y **demostrar** qué pasa cuando se rompe la propagación.
5. **Correlacionar** señales: de una métrica anómala a una traza (exemplar) y de la traza a sus logs.
6. **Aplicar** las *golden signals* (y RED/USE) a un agente: latencia, tráfico, errores y saturación **más** tokens/coste, pasos y calidad.
7. **Calcular** por qué los percentiles no se promedian y cómo **explota la cardinalidad** de una métrica.
8. **Decidir** qué señal detecta mejor cada tipo de fallo de un agente (preparación del proyecto).

## 🔁 Conexión con módulos anteriores

<details>
<summary>1 · (Módulo 00) ¿Qué trae el Collector en una petición OTLP/HTTP de trazas: dónde van `traceId` y los tiempos y en qué formato?</summary>

En `POST :4318/v1/traces`, JSON jerárquico `resourceSpans → scopeSpans → spans`; `traceId` (32 hex) y `spanId` (16 hex) en **hexadecimal**, y `startTimeUnixNano`/`endTimeUnixNano` como **strings** de nanosegundos.

</details>

<details>
<summary>2 · (Módulo 01) ¿Por qué una alucinación es *más barata y rápida* que una respuesta correcta y qué implica para el monitoreo?</summary>

Porque no llama herramientas (1 paso, ≈ 350 tokens frente a 5 pasos, ≈ 1300). Implica que una «mejora» de latencia/coste puede ser una regresión: hay que vigilar pasos/herramientas por petición y la calidad, no solo latencia y coste.

</details>

<details>
<summary>3 · (Módulo 01) ¿Qué punto de extensión del agente permite instrumentarlo y qué tres operaciones envuelve?</summary>

El `Instrumentor` con context managers: `agent_run` (span raíz `invoke_agent`), `llm_call` (`chat`) y `tool_call` (`execute_tool`), más `log(...)`.

</details>

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz]"], check=True)
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

_ = check_stack([])      # este módulo no necesita servicios
import json, re, secrets, bisect, math

## Los datos de esta lección

Trabajaremos con dos fuentes, ambas con el **mismo esquema** que producirá el SDK de OpenTelemetry:

- **Pregrabados** (`sample_data`): 240 tickets de 60 min con fallos de fondo y **dos incidentes** (*proveedor LLM degradado* en los minutos 20-30 y *herramientas lentas* en 40-48): ejecuciones (`runs`), *spans* y *logs*.
- **Generados al vuelo** con `RecordingInstrumentor`, para cuando necesitemos ver algo concreto.

In [ ]:
runs = pd.DataFrame(sample_data.load_runs())
spans = pd.DataFrame(sample_data.load_spans())
logs = pd.DataFrame(sample_data.load_logs())
meta = sample_data.load_meta()
spans["dur"] = (spans.end_unix_nano - spans.start_unix_nano) / 1e9
spans["minute"] = ((spans.start_unix_nano / 1e9 - SIM_EPOCH_S) // 60).astype(int)
runs["minute"] = ((runs.created_at - SIM_EPOCH_S) // 60).astype(int)
print(f"{len(runs)} ejecuciones · {len(spans)} spans · {len(logs)} logs · incidentes: {[ (i['name'], i['start_s']//60, i['end_s']//60) for i in meta['incidents']]}")
spans.head(3)[["trace_id", "span_id", "parent_span_id", "name", "kind", "status", "dur"]]

---
# 1 · Intuición: tres preguntas, tres señales

Imagina que un cliente dice «el bot tardó una eternidad». Para investigarlo haces tres preguntas distintas:

| Pregunta | Señal | Forma de los datos | Coste por dato | Para qué sirve |
|---|---|---|---|---|
| **¿Qué pasó exactamente?** | **Logs** | Eventos discretos con texto y campos | Alto (cada evento) | Detalle, auditoría, «qué dijo el error» |
| **¿Cuánto y con qué frecuencia?** | **Métricas** | Series temporales numéricas agregadas | **Bajo y constante** | Alertas, tendencias, SLOs |
| **¿Por dónde fue y dónde se fue el tiempo?** | **Trazas** | Árbol de *spans* de una petición | Alto (por petición, se muestrea) | Depurar un caso: causalidad y latencia |

**Analogías:**

| Si vienes de… | Logs | Métricas | Trazas |
|---|---|---|---|
| **ML** | Predicción *por muestra* (entrada, salida, error) | Métrica agregada (accuracy, AUC por lote) | *Debugging* de una predicción concreta: qué *features* y qué pasos del pipeline la produjeron |
| **Microservicios** | `logger.error(...)` en cada servicio | Latencia p95, RPS, tasa de errores por endpoint | El *tracing distribuido* (Jaeger/Zipkin) de una petición a través de servicios |
| **SRE** | La evidencia para el postmortem | Los SLIs que alimentan el SLO y la alerta | La herramienta del *on-call* para pasar de «algo va lento» a «esto va lento» |

Veamos **las tres vistas de una misma petición lenta** de SupportOps, y cómo se *complementan*:

In [ ]:
demo_tk = generate_tickets(120, seed=SEED, fault_rates=DEFAULT_FAULT_RATES)
rec = RecordingInstrumentor()
demo_res = run_tickets(SupportAgent(instrumentor=rec), demo_tk)
roots = [s for s in rec.spans if s["parent_span_id"] is None]
warned = {l["trace_id"] for l in rec.logs if l["severity_number"] >= 13}                # trazas con algún log WARN o peor
slow = max([s for s in roots if s["trace_id"] in warned] or roots, key=lambda s: s["end_unix_nano"] - s["start_unix_nano"])
tid = slow["trace_id"]
mine = [s for s in rec.spans if s["trace_id"] == tid]
my_logs = [l for l in rec.logs if l["trace_id"] == tid]
D = pd.DataFrame(to_rows(demo_tk, demo_res)); D["minute"] = ((D.created_at - SIM_EPOCH_S) // 60).astype(int)
t0 = mine[0]["start_unix_nano"]; slow_minute = int((t0 / 1e9 - SIM_EPOCH_S) // 60)

fig = plt.figure(figsize=(15, 5.2)); gs = fig.add_gridspec(2, 2, width_ratios=[1.5, 1], left=0.19, right=0.98, wspace=0.12, hspace=0.45)
ax1 = fig.add_subplot(gs[:, 0]); viz.waterfall(mine, ax=ax1, title="TRAZAS · ¿por dónde fue y dónde se fue el tiempo? (1 petición)")
ax2 = fig.add_subplot(gs[0, 1]); ax2.axis("off"); ax2.set_title("LOGS · ¿qué pasó? (eventos de esa petición)", loc="left")
ax2.text(0, 0.9, "\n".join(f"{(l['time_unix_nano'] - t0)/1e9:5.2f}s {l['severity_text']:5s} {l['body']}" for l in my_logs), va="top", family="monospace", fontsize=8.5)
ax3 = fig.add_subplot(gs[1, 1]); per_min = D.groupby("minute").latency_s.agg(["count", lambda x: np.quantile(x, .95)])
ax3.bar(per_min.index, per_min["count"], color=viz.PALETTE["metrics"], alpha=.5, label="peticiones/min"); ax3.set_ylabel("peticiones/min")
ax3b = ax3.twinx(); ax3b.plot(per_min.index, per_min.iloc[:, 1], color=viz.PALETTE["error"], marker="o", ms=3, label="p95 (s)"); ax3b.set_ylabel("p95 latencia (s)"); ax3b.grid(False)
ax3.axvline(slow_minute, color="k", ls=":"); ax3.set_title("MÉTRICAS · ¿cuánto? (todas las peticiones, agregadas)", loc="left"); ax3.set_xlabel("minuto")
plt.show()
print(f"trace_id de la petición lenta: {tid}  ({(slow['end_unix_nano']-slow['start_unix_nano'])/1e9:.1f} s)")

> 👀 **Qué debes observar**
>
> - **Métricas (abajo-derecha):** ven *que* hay un problema (el p95 sube en ese minuto) pero **no dicen cuál petición ni por qué**: la información se perdió al agregar.
> - **Trazas (izquierda):** muestran la **estructura** de esa petición concreta —cuántos pasos hubo, cuál fue lento, cuál falló (en rojo)— y por tanto **dónde** se fue el tiempo.
> - **Logs (arriba-derecha):** cuentan **qué ocurrió** con sus palabras (el mensaje de error), enlazados a la traza por el mismo identificador.
> - Se complementan: la métrica *avisa*, la traza *localiza*, el log *explica*. Ninguna sustituye a las otras; el resto del módulo es cómo se construyen y se unen.

---
# 2 · Las tres señales «a mano»

Antes del SDK, construimos cada señal con Python puro para entender qué datos contiene y qué *pierde*.

## 2.1 Logs: un evento con campos

Un log es *(instante, severidad, mensaje, atributos)*. La diferencia clave es **texto libre vs estructurado**:

In [ ]:
l = logs[logs.body == "error de herramienta"].iloc[0]
print("NO estructurado (hay que parsear con regex):")
print("  ", f"{l.severity_text} {l.body} " + " ".join(f"{k}={v}" for k, v in l.attributes.items()))
print("\nEstructurado (cada campo consultable):")
print(json.dumps({"time_unix_nano": int(l.time_unix_nano), "severity_text": l.severity_text, "severity_number": int(l.severity_number), "body": l.body,
                  "attributes": l.attributes, "trace_id": l.trace_id, "span_id": l.span_id}, ensure_ascii=False, indent=1)[:620])
# Pregunta analítica: ¿qué herramienta falla más? Con atributos es una línea; con texto sería un regex frágil.
w = logs[logs.body == "error de herramienta"]
print("\nfallos por herramienta:", w.attributes.map(lambda a: a.get("tool")).value_counts().to_dict(), "·", "por error_type:", w.attributes.map(lambda a: a.get("error_type")).value_counts().to_dict())

Dos propiedades hacen valiosos a los logs de producción: **estructura** (campos con nombre) y **correlación** (`trace_id` y `span_id` del span activo cuando se emitió el log). Severidad según OpenTelemetry:
`1-4 TRACE`, `5-8 DEBUG`, `9-12 INFO`, `13-16 WARN`, `17-20 ERROR`, `21-24 FATAL`.

## 2.2 Métricas: contador, *gauge* e histograma

Una métrica es una **serie temporal de números agregados**; no guarda peticiones individuales. Tres instrumentos esenciales (hay más: *UpDownCounter*, asíncronos…):

| Instrumento | Qué es | Ejemplo en SupportOps | Operación válida |
|---|---|---|---|
| **Counter** | Solo sube (o se reinicia a 0 al reiniciar el proceso) | `agent_requests_total{status}` | `rate()`, `increase()` |
| **Gauge** | Valor actual, sube y baja | ejecuciones en curso; uso de la ventana de contexto | media, máx., último |
| **Histogram** | Distribución en *buckets* + suma + cuenta | `agent_request_duration_seconds` | percentiles aproximados, agregables entre instancias |

Los implementamos **con su salida en el formato de texto de Prometheus** (lo que vería `GET /metrics`):

In [ ]:
class Counter:
    def __init__(self, name): self.name, self.v = name, {}
    def inc(self, n=1, **labels): k = tuple(sorted(labels.items())); self.v[k] = self.v.get(k, 0) + n

class Gauge:
    def __init__(self, name): self.name, self.v = name, {}
    def set(self, x, **labels): self.v[tuple(sorted(labels.items()))] = x

class Histogram:
    """Buckets con límites superiores (le = «menor o igual»); el último bucket implícito es +Inf."""
    def __init__(self, name, bounds): self.name, self.bounds = name, list(bounds); self.counts = [0] * (len(bounds) + 1); self.sum = 0.0; self.count = 0
    def record(self, x):
        self.counts[bisect.bisect_left(self.bounds, x)] += 1; self.sum += x; self.count += 1
    def quantile(self, q):
        """Estimación por interpolación lineal dentro del bucket (la misma idea que histogram_quantile de PromQL)."""
        target, cum = q * self.count, 0
        for i, c in enumerate(self.counts):
            if c and cum + c >= target:
                lo = 0.0 if i == 0 else self.bounds[i - 1]
                hi = self.bounds[i] if i < len(self.bounds) else self.bounds[-1] * 2        # el bucket +Inf no tiene techo: aproximamos
                return lo + (hi - lo) * (target - cum) / c
            cum += c
        return float("nan")

def exposition(*metrics):
    out = []
    for m in metrics:
        if isinstance(m, Histogram):
            out.append(f"# TYPE {m.name} histogram"); cum = 0
            for b, c in zip(m.bounds + [float("inf")], m.counts):
                cum += c; le = "+Inf" if b == float("inf") else b
                out.append(m.name + '_bucket{le="' + str(le) + '"} ' + str(cum))
            out += [f"{m.name}_sum {m.sum:.3f}", f"{m.name}_count {m.count}"]
        else:
            out.append(f"# TYPE {m.name} " + ("counter" if isinstance(m, Counter) else "gauge"))
            for k, v in m.v.items():
                lab = ",".join(a + '="' + str(b) + '"' for a, b in k)
                out.append(m.name + "{" + lab + "} " + str(v))
    return "\n".join(out)

req_total = Counter("agent_requests_total"); dur = Histogram("agent_request_duration_seconds", [0.5, 1, 2, 3, 5, 8, 13]); ctx_gauge = Gauge("agent_context_tokens")
for r in runs.itertuples():
    req_total.inc(status=r.status); dur.record(r.latency_s); ctx_gauge.set(r.input_tokens, intent=r.intent_true)     # último valor por intención
print(exposition(req_total, dur, ctx_gauge))

### El histograma y el error de los percentiles

Un histograma **descarta los valores individuales** y solo conserva *cuántos* cayeron en cada bucket. El percentil se estima **interpolando linealmente** dentro del bucket que contiene el rango $q\cdot N$:

$$ \hat{x}_q \;=\; L_i \;+\; (U_i-L_i)\,\frac{q\,N-C_{i-1}}{c_i} $$

donde $[L_i,U_i]$ son los límites del bucket $i$ que contiene el percentil, $c_i$ su recuento y $C_{i-1}$ el recuento acumulado de los buckets anteriores. El **error** depende de cuán anchos son los buckets *justo donde está el percentil*.

In [ ]:
layouts = {"3 buckets gruesos [2, 10]": [2, 10], "exponencial [0.5…16]": [0.5, 1, 2, 4, 8, 16], "fino, paso 0.25 s": list(np.arange(0.25, 15, 0.25))}
exact = {q: np.quantile(runs.latency_s, q) for q in (0.5, 0.95)}
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=False)
for ax, (name, b) in zip(axes, layouts.items()):
    h = Histogram("x", b); [h.record(v) for v in runs.latency_s]
    edges = [0] + h.bounds + [h.bounds[-1] * 1.6]; widths = np.diff(edges)
    ax.bar(edges[:-1], h.counts, width=widths, align="edge", color=viz.PALETTE["traces"], alpha=.7, edgecolor="white")
    for q, c in ((0.5, "ok"), (0.95, "error")):
        ax.axvline(h.quantile(q), color=viz.PALETTE[c], ls="--", lw=2); ax.axvline(exact[q], color="k", ls=":", lw=1.5)
    ax.set_title(f"{name}\np95 estimado={h.quantile(.95):.2f}s · exacto={exact[0.95]:.2f}s (err {abs(h.quantile(.95)/exact[0.95]-1):.0%})", fontsize=10)
    ax.set_xlabel("latencia (s)"); ax.set_xlim(0, 16)
axes[0].set_ylabel("peticiones por bucket")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - Las líneas **discontinuas de color** son la estimación desde los buckets (verde = mediana, roja = p95); las **punteadas negras**, el valor exacto calculado con todos los datos.
> - Con 3 buckets gruesos el p95 puede errar mucho (la estimación interpola dentro de un bucket enorme). Con buckets **exponenciales** el error baja bastante y con buckets finos casi desaparece… a cambio de **más series** (cada bucket es una serie en Prometheus).
> - Regla práctica de la élite: **pon límites de bucket alrededor de tus umbrales de SLO** (p. ej. justo en 3 s si tu SLO es «95 % < 3 s»), porque ahí es donde necesitas exactitud.

## 2.3 Trazas: spans con padre, y «tiempo propio»

Un **span** es una operación con inicio y fin; los spans de una petición forman un **árbol** (cada hijo apunta a su padre con `parent_span_id`). La traza es el conjunto de spans con el mismo `trace_id`.
Una medida útil es el **tiempo propio** (*self time*): lo que tarda un span **sin contar a sus hijos**. Para un agente responde a «¿dónde se va el tiempo?».

In [ ]:
child_sum = spans[spans.parent_span_id.notna()].groupby("parent_span_id").dur.sum()
spans["self"] = spans.dur - spans.span_id.map(child_sum).fillna(0)
spans["tipo"] = spans.name.str.split().str[0].map({"invoke_agent": "agente (reintentos, esperas)", "chat": "LLM (chat)", "execute_tool": "herramientas"})
st = runs.set_index("ticket_id").status
tr2status = spans[spans.parent_span_id.isna()].assign(t=lambda d: d.attributes.map(lambda a: a["supportops.ticket.id"])).set_index("trace_id").t.map(st)
spans["estado"] = spans.trace_id.map(tr2status)
pivot = spans.pivot_table(index="estado", columns="tipo", values="self", aggfunc="sum").fillna(0)
share = pivot.div(pivot.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(9, 3.8))
left = np.zeros(len(share))
for col, c in zip(["LLM (chat)", "herramientas", "agente (reintentos, esperas)"], [viz.PALETTE["traces"], viz.PALETTE["logs"], viz.PALETTE["warn"]]):
    ax.barh(share.index, share[col], left=left, color=c, label=col); left += share[col].values
ax.set_xlabel("% del tiempo (suma de tiempo propio de los spans)"); ax.set_title("¿Dónde se va el tiempo? Según cómo terminó la petición"); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, fontsize=8)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - En las peticiones `ok` **el LLM se lleva ≈ 90 % del tiempo**: optimizar una API interna casi no mueve la latencia de extremo a extremo.
> - En `degraded` (herramienta fallida o lenta) **las herramientas pasan de ≈ 10 % a ≈ 30 %** del tiempo (los timeouts de 5 s). En `error` aparece un ≈ 40 % de tiempo **del propio agente** (esperas de reintento con *backoff* mientras el proveedor falla).
> - Esta es una pregunta que **solo las trazas pueden contestar**: una métrica de latencia por separado no sabe qué parte de la petición pertenece a qué componente.

---
# 3 · El modelo de datos de OpenTelemetry

OpenTelemetry define un **modelo común** para las tres señales (y el protocolo OTLP que lo transporta). Todas comparten la misma envoltura: **quién** emite (*resource*) y **qué librería** lo midió (*scope*).

- **Resource**: entidad que produce telemetría; un conjunto de atributos **inmutable**, p. ej. `service.name`, `service.version`, `deployment.environment.name`. Se comparte por todas las señales: es la clave de *join* gratuita.
- **InstrumentationScope**: nombre (y versión) de la librería o módulo instrumentado (p. ej. `supportops.agent`). Permite saber *quién* instrumentó (y apagarlo).

In [ ]:
fig, ax = viz.canvas(13, 5.4, (0, 100), (0, 44))
viz.box(ax, 1, 1, 98, 41, "", viz.PALETTE["neutral"], fc="#00000004"); ax.text(2.5, 39.5, "ResourceSpans  (1 por resource)", fontsize=10, fontweight="bold", color=viz.PALETTE["neutral"])
viz.box(ax, 3, 24, 28, 12, "Resource\nservice.name = supportops-agent\nservice.version = 0.1.0\ndeployment.environment.name = lab", viz.PALETTE["neutral"], fontsize=8)
viz.box(ax, 34, 3, 63, 34, "", viz.PALETTE["traces"], fc="#2a6fdb08"); ax.text(35.5, 34.5, "ScopeSpans  (1 por instrumentation scope)", fontsize=10, fontweight="bold", color=viz.PALETTE["traces"])
viz.box(ax, 36, 24, 24, 7, "Scope\nname = supportops.agent", viz.PALETTE["traces"], fontsize=8)
viz.box(ax, 36, 5, 28, 17, "Span\ntrace_id (16 B) · span_id (8 B)\nparent_span_id\nname · kind\nstart/end (ns)\nstatus (Unset|Ok|Error)", viz.PALETTE["traces"], fontsize=8, bold=False)
viz.box(ax, 67, 14, 28, 9, "attributes (clave→valor tipado)\ngen_ai.*, supportops.*", viz.PALETTE["metrics"], fontsize=8)
viz.box(ax, 67, 4.5, 13, 7, "events\n(exception…)", viz.PALETTE["logs"], fontsize=8); viz.box(ax, 82, 4.5, 13, 7, "links\n(otras trazas)", viz.PALETTE["profiles"], fontsize=8)
viz.arrow(ax, (64.4, 15), (66.6, 17.5)); viz.arrow(ax, (64.4, 10), (66.6, 8))
ax.text(3, 15, "Mismo esquema para las otras señales:\nLogs: ResourceLogs → ScopeLogs → LogRecord\nMétricas: ResourceMetrics → ScopeMetrics → Metric", fontsize=8.5, style="italic", va="top")
plt.show()

> 👀 **Qué debes observar**
>
> - **Anidamiento**: `Resource ⊃ Scope ⊃ Span`. Lo que es común (quién emite, qué librería) se escribe **una vez** por lote, no por span; por eso OTLP es compacto.
> - Los atributos de **resource** describen *al emisor* y viajan con **todas** las señales (puedes filtrar logs, métricas y trazas por `service.name`); los atributos de **span** describen *esa operación*.
> - Logs y métricas tienen la misma estructura envolvente. Es lo que hizo posible nuestro «hola mundo» de 3 señales del módulo 00.

## 3.1 El span, con precisión

| Campo | Contenido | Notas |
|---|---|---|
| `trace_id` | 16 bytes (32 hex) | Al menos un byte distinto de cero. Común a toda la traza |
| `span_id` | 8 bytes (16 hex) | Único dentro de la traza |
| `parent_span_id` | `span_id` del padre | Vacío en el span **raíz** |
| `name` | Texto de **baja cardinalidad** | `execute_tool get_order`, **no** `execute_tool ORD-00012` |
| `kind` | `INTERNAL` (por defecto), `CLIENT`, `SERVER`, `PRODUCER`, `CONSUMER` | `CLIENT`: petición saliente que espera respuesta (p. ej. llamada al LLM); `SERVER`: la recibe; `PRODUCER/CONSUMER`: ejecución diferida (colas) |
| `start/end` | ns desde la época Unix | Dos relojes: usa el monotónico para durar (ver 🧠) |
| `status` | `Unset` (defecto), `Ok`, `Error` (+ mensaje) | Orden total `Ok > Error > Unset`; las librerías **no deberían** poner `Ok`; las aplicaciones sí pueden |
| `attributes` | Pares clave→valor tipados | La carne del análisis: modelo, tokens, herramienta, ticket |
| `events` | Marcas con tiempo dentro del span | Excepciones, «reintento»; baratos y viajan con la traza |
| `links` | Referencias a otros spans/trazas | Relaciones no padre-hijo (p. ej. un lote que procesa muchos mensajes) |

Veamos nuestros spans pregrabados **convertidos a OTLP/JSON a mano** (la misma forma que enviaste al Collector en el módulo 00):

In [ ]:
def to_otlp_json(span_records):
    """Spans (formato SPAN_SCHEMA del curso) → documento OTLP/JSON `resourceSpans`. Un resource, un scope."""
    def val(v):
        if isinstance(v, bool): return {"boolValue": v}
        if isinstance(v, int): return {"intValue": str(v)}
        if isinstance(v, float): return {"doubleValue": v}
        if isinstance(v, list): return {"arrayValue": {"values": [val(x) for x in v]}}
        return {"stringValue": str(v)}
    code_of = {"UNSET": 0, "OK": 1, "ERROR": 2}; kind_of = {"INTERNAL": 1, "SERVER": 2, "CLIENT": 3}
    res = span_records[0]["resource"]
    return {"resourceSpans": [{"resource": {"attributes": [{"key": k, "value": val(v)} for k, v in res.items()]},
        "scopeSpans": [{"scope": {"name": span_records[0]["scope"]}, "spans": [
            {"traceId": s["trace_id"], "spanId": s["span_id"], **({"parentSpanId": s["parent_span_id"]} if s["parent_span_id"] else {}),
             "name": s["name"], "kind": kind_of[s["kind"]], "startTimeUnixNano": str(s["start_unix_nano"]), "endTimeUnixNano": str(s["end_unix_nano"]),
             "attributes": [{"key": k, "value": val(v)} for k, v in s["attributes"].items()],
             "status": {"code": code_of[s["status"]], **({"message": s["status_message"]} if s["status_message"] else {})}} for s in span_records]}]}]}

one_trace = [s for s in sample_data.load_spans() if s["trace_id"] == sample_data.load_spans()[40]["trace_id"]]
doc = to_otlp_json(one_trace)
print(json.dumps(doc["resourceSpans"][0]["scopeSpans"][0]["spans"][1], indent=1, ensure_ascii=False)[:900])
print(f"\n{len(one_trace)} spans → {len(json.dumps(doc))} bytes de JSON")

> ⚠️ **No envíes estos datos tal cual a un Tempo vivo**: los timestamps simulados son de septiembre de 2026 y los backends descartan o rechazan datos muy antiguos. Para enviar tráfico en vivo, `generate_tickets(..., start_s="live")`.

## 3.2 Contexto y propagación (W3C Trace Context)

¿Cómo sabe un servicio B que su trabajo es *hijo* del span de A, si están en procesos distintos? **A le envía un identificador en una cabecera**. El estándar es la cabecera **`traceparent`** de W3C Trace Context:

```text
traceparent: 00-4bf92f3577b34da6a3ce929d0e0e4736-00f067aa0ba902b7-01
             │  └──────── trace-id (32 hex) ───┘ └─ parent-id (16 hex) ┘ └ flags
             └ versión
```

El último byte son *flags*; el bit `01` = **sampled** («esta traza se está registrando»). El receptor extrae `trace_id` y `parent-id`, crea su propio span con ese **padre**, y sigue propagando. Si **nadie** propaga (o se pierde en un proxy/cola), la traza se **rompe** en dos.
Implementémoslo a mano y simulemos un orquestador que llama a un servidor de herramientas (esto es lo que harán los módulos 11 y 12):

In [ ]:
def make_traceparent(trace_id: str, span_id: str, sampled: bool = True) -> str:
    return f"00-{trace_id}-{span_id}-{'01' if sampled else '00'}"

def parse_traceparent(h: str | None):
    """Devuelve dict o None si es inválida (formato incorrecto o ids todo ceros, que la spec declara inválidos)."""
    m = re.fullmatch(r"([0-9a-f]{2})-([0-9a-f]{32})-([0-9a-f]{16})-([0-9a-f]{2})", (h or "").strip())
    if not m or set(m.group(2)) == {"0"} or set(m.group(3)) == {"0"}: return None
    v, t, s, f = m.groups(); return {"version": v, "trace_id": t, "parent_id": s, "sampled": bool(int(f, 16) & 1)}

SIM = {"t": 0.0, "spans": []}                                       # reloj y almacén de spans del mini-experimento
def new_span(name, kind, trace_id=None, parent=None, dur=0.1, service="?"):
    s = {"trace_id": trace_id or secrets.token_hex(16), "span_id": secrets.token_hex(8), "parent_span_id": parent, "name": name, "kind": kind, "service": service,
         "status": "OK", "start_unix_nano": int(SIM["t"] * 1e9), "end_unix_nano": int((SIM["t"] + dur) * 1e9)}
    SIM["spans"].append(s); return s

def tool_server(headers: dict):                                     # «otro proceso»: solo ve las cabeceras que le llegan
    ctx = parse_traceparent(headers.get("traceparent"))
    SIM["t"] += 0.05
    return new_span("execute_tool get_order (servidor)", "SERVER", trace_id=ctx["trace_id"] if ctx else None, parent=ctx["parent_id"] if ctx else None, dur=0.12, service="orders-mcp")

def orchestrator(propagate: bool):
    SIM.update(t=0.0, spans=[])
    root = new_span("invoke_agent supportops", "INTERNAL", dur=0.6, service="supportops-agent")
    call = new_span("execute_tool get_order (cliente)", "CLIENT", trace_id=root["trace_id"], parent=root["span_id"], dur=0.25, service="supportops-agent")
    headers = {"traceparent": make_traceparent(call["trace_id"], call["span_id"])} if propagate else {}      # inyectar en la petición saliente
    tool_server(headers)
    return list(SIM["spans"])

ok, roto = orchestrator(True), orchestrator(False)
print("con propagación :", len({s["trace_id"] for s in ok}), "traza(s) ·", sum(s["parent_span_id"] is None for s in ok), "raíz")
print("sin propagación :", len({s["trace_id"] for s in roto}), "traza(s) ·", sum(s["parent_span_id"] is None for s in roto), "raíces (huérfano)")
print("round-trip traceparent:", parse_traceparent(make_traceparent("ab"*16, "cd"*8)), "| inválida →", parse_traceparent("00-000-xx"))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6)); fig.subplots_adjust(left=0.3, hspace=0.6)
for ax, data, title in [(axes[0], ok, "CON propagación: 1 traza, árbol completo"), (axes[1], roto, "SIN propagación: 2 trazas, el servidor queda huérfano")]:
    viz.waterfall(data, ax=ax, title=title)
plt.show()

> 👀 **Qué debes observar**
>
> - Con la cabecera `traceparent`, el span del servidor (`SERVER`) cuelga del span cliente (`CLIENT`): **un solo árbol**, con la causalidad correcta.
> - Sin ella, el servidor crea **otra traza** (otro `trace_id`) con su propio span raíz: en Tempo verías *dos* trazas inconexas y no podrías contestar «¿cuánto tardó el servidor de pedidos *dentro* de esta petición?».
> - En un agente multi-proceso (orquestador + subagentes + servidores MCP + colas) **cada salto** debe inyectar y extraer el contexto; un solo eslabón roto parte la traza. Es la causa nº 1 de «mis trazas salen cortadas».

## 3.3 Métricas: temporalidad y tasas

Un **contador** puede transmitirse como **acumulado** (*cumulative*: «desde el arranque llevo 4812 peticiones») o como **delta** («en los últimos 10 s hubo 37»). Prometheus usa acumulado y obtiene la **tasa** con `rate()` = incremento / tiempo.
Si el proceso **se reinicia**, el contador vuelve a 0: una resta ingenua daría un valor *negativo*; `rate()` detecta la caída y la trata como reinicio.

In [ ]:
per_min = runs.groupby("minute").size().reindex(range(60), fill_value=0)
cum = per_min.cumsum().values.astype(float)
RESTART = 35                                                    # el proceso reinicia en el minuto 35: el contador vuelve a 0
obs = cum.copy(); obs[RESTART:] = cum[RESTART:] - cum[RESTART - 1]
naive = np.diff(obs, prepend=obs[0])                            # resta ingenua
def rate_reset_aware(c):
    d = np.diff(c, prepend=c[0]); return np.where(d < 0, c, d)  # si baja → reinicio: el incremento es el valor actual
smart = rate_reset_aware(obs)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(obs, color=viz.PALETTE["metrics"], lw=2); ax[0].axvline(RESTART, color="k", ls=":"); ax[0].text(RESTART + .5, obs.max() * .75, "reinicio del proceso", fontsize=9)
ax[0].set_title("Contador ACUMULADO agent_requests_total"); ax[0].set_xlabel("minuto"); ax[0].set_ylabel("valor del contador")
ax[1].bar(np.arange(60) - .2, naive, .4, color=viz.PALETTE["error"], label="resta ingenua (¡negativa!)"); ax[1].bar(np.arange(60) + .2, smart, .4, color=viz.PALETTE["ok"], label="tratando reinicios (rate)")
ax[1].set_title("Peticiones por minuto"); ax[1].set_xlabel("minuto"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"incremento ingenuo en el minuto del reinicio: {naive[RESTART]:.0f}  ·  con tratamiento de reinicios: {smart[RESTART]:.0f}  ·  real: {per_min[RESTART]}")

> 👀 **Qué debes observar**
>
> - A la izquierda, el contador cae a casi 0 en el reinicio. A la derecha, la **resta ingenua** produce un pico negativo enorme (rojo); el tratamiento de reinicios (verde) recupera la tasa real.
> - Moraleja: **nunca restes contadores a mano**: usa `rate()`/`increase()`. Y recuerda que la tasa se calcula *por ventana*: ventanas cortas = ruidosas, largas = lentas en reaccionar.
> - Un contador **delta** (OTLP permite elegir) evita este problema pero obliga al backend a convertir a acumulado si lo necesita (Prometheus lo hace con `otlp-deltatocumulative`).

## 3.4 Logs: el registro y su correlación

| Campo del `LogRecord` | Significado |
|---|---|
| `Timestamp` / `ObservedTimestamp` | Cuándo ocurrió / cuándo lo observó el sistema de recogida |
| `SeverityNumber` / `SeverityText` | Nivel (1-24) y su nombre |
| `Body` | El mensaje (texto o estructura) |
| `Attributes` | Campos estructurados |
| **`TraceId`, `SpanId`, `TraceFlags`** | **Correlación con la traza activa** cuando se emitió el log |
| `Resource`, `InstrumentationScope` | Quién y con qué librería |

## 3.5 Resumen: los cuatro «ejes» de cada señal

| | Logs | Métricas | Trazas |
|---|---|---|---|
| Unidad | Evento | Punto agregado (serie × tiempo) | Span (operación) en un árbol |
| Cardinalidad que soporta | Alta (cada evento es único) | **Baja** (cada combinación de labels = una serie) | Alta (cada petición es única) |
| Coste | ∝ eventos | ∝ **series** (no eventos) | ∝ peticiones muestreadas |
| Retiene detalle por petición | Sí | **No** | Sí |
| Alerta | Posible pero cara | **La mejor opción** | Rara (se usa para investigar) |

---
# 4 · Correlacionar: de la métrica a la traza y de la traza a los logs

El valor máximo aparece cuando **las señales se enlazan**. Hay tres claves de unión:

1. **Recurso común** (`service.name`, versión, entorno): une señales *del mismo servicio*.
2. **`trace_id` / `span_id` en los logs**: une un log con *su* traza (y su span).
3. **Exemplars**: un punto de una métrica (p. ej. el bucket más lento de un histograma) guarda el `trace_id` **de una petición concreta** que contribuyó a ese valor: el puente métrica → traza.

Recorramos el camino completo con el incidente de los minutos 20-30 (proveedor LLM degradado):

In [ ]:
roots = spans[spans.parent_span_id.isna()].copy()
roots["ticket"] = roots.attributes.map(lambda a: a["supportops.ticket.id"])
roots = roots.merge(runs[["ticket_id", "latency_s", "status", "minute"]].rename(columns={"status": "run_status", "minute": "run_minute"}), left_on="ticket", right_on="ticket_id")
pm = roots.groupby("run_minute").agg(p95=("latency_s", lambda x: np.quantile(x, .95)), n=("latency_s", "size"), err=("run_status", lambda s: s.isin(["error", "max_steps"]).mean()))
# EXEMPLAR: por minuto, la traza más lenta (la que un histograma guardaría como exemplar del bucket más alto)
ex = roots.loc[roots.groupby("run_minute").latency_s.idxmax()].set_index("run_minute")
worst_min = pm.err.idxmax(); ex_tid = ex.loc[worst_min, "trace_id"]
fig = plt.figure(figsize=(11, 7.6)); gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.1], left=0.3, right=0.97, hspace=0.45)
ax1 = fig.add_subplot(gs[0]); ax1.plot(pm.index, pm.err * 100, color=viz.PALETTE["error"], marker="o", ms=3, label="% errores"); ax1.set_ylabel("% de peticiones con error", color=viz.PALETTE["error"]); ax1.set_xlabel("minuto")
ax1.axvspan(20, 30, color=viz.PALETTE["error"], alpha=.1); ax1.axvspan(40, 48, color=viz.PALETTE["warn"], alpha=.12)
ax1.scatter([worst_min], [pm.err[worst_min] * 100], s=120, facecolors="none", edgecolors="k", zorder=5); ax1.annotate("exemplar →", (worst_min, pm.err[worst_min] * 100), (worst_min - 14, pm.err[worst_min] * 100 + 3), arrowprops=dict(arrowstyle="->"))
ax1.set_title("MÉTRICA: tasa de error por minuto (franjas: incidentes)")
ax2 = fig.add_subplot(gs[1]); viz.waterfall(spans[spans.trace_id == ex_tid].to_dict("records"), ax=ax2, title=f"TRAZA del exemplar (minuto {worst_min})")
plt.show()
print("LOGS de esa traza (join por trace_id):")
for r in logs[logs.trace_id == ex_tid].itertuples(): print(f"  {r.severity_text:5s} {r.body}  {r.attributes}")

> 👀 **Qué debes observar**
>
> - La métrica señala un pico (**≈ minuto 26, dentro de la franja roja**): una tasa de error muy alta. Eso es «*que* algo pasa».
> - El **exemplar** nos lleva a *una* traza concreta de ese minuto: ahí vemos el span `chat` en rojo (el proveedor devolvió 5xx), los reintentos y el agente rindiéndose. Eso es «*dónde*».
> - Los **logs unidos por `trace_id`** dicen «error del proveedor LLM» con `status=503`. Eso es «*qué*».
> - Fíjate en la franja naranja (minutos 40-48): ahí **no hay errores** (la métrica de error no se mueve), pero algo pasa con la latencia: hay que mirar otras señales. Lo vemos en la sección 5.

---
# 5 · Las *golden signals* aplicadas a agentes

El libro de SRE de Google propone cuatro señales para monitorizar cualquier sistema orientado a usuarios: **latencia, tráfico, errores y saturación**. Dos recetas relacionadas:
**RED** (*Rate, Errors, Duration*; Tom Wilkie, pensada para microservicios) y **USE** (*Utilization, Saturation, Errors*; Brendan Gregg, pensada para *recursos*).
Para un agente hay que **extenderlas**:

| Señal | En un servicio clásico | En un agente (SupportOps) | Dónde se mide |
|---|---|---|---|
| **Latencia** | Duración de la petición | Duración de `invoke_agent` **y** de cada `chat`/`execute_tool`; si hay *streaming*, tiempo al primer token. Separar éxito de error | Histograma + trazas |
| **Tráfico** | Peticiones/s | Peticiones/s **y tokens/s** (el tráfico real del proveedor es en tokens) | Contador |
| **Errores** | `5xx`, excepciones | Errores explícitos (**`error`**, `max_steps`) **+ implícitos**: respuesta inventada o con PII, herramienta caída con respuesta de cortesía (`degraded`) | Contador por `status`/`error.type` + evaluación online |
| **Saturación** | CPU, colas, hilos | Límites de tasa del proveedor (429), peticiones concurrentes, **uso de la ventana de contexto**, cola de trabajos | Gauge / contadores de 429 |
| **+ Coste** | (≈ constante) | USD por petición / por cliente / por función | Contador de tokens × precio |
| **+ Pasos** | 1 | Llamadas LLM + herramientas por petición (¡bucles!) | Histograma |
| **+ Calidad** | (implícita) | Fundamentación, utilidad, seguridad (módulo 14) | Evaluadores online |

Construyamos el «dashboard de oro» de SupportOps con los datos pregrabados:

In [ ]:
runs["steps"] = runs.n_llm_calls + runs.n_tool_calls
g = runs.groupby("minute")
M = pd.DataFrame({
    "tráfico (peticiones/min)": g.size().reindex(range(60), fill_value=0),
    "errores (%)": (g.status.apply(lambda x: x.isin(["error", "max_steps"]).mean()) * 100).reindex(range(60)),
    "latencia p50 (s)": pd.Series([runs[(runs.minute <= m) & (runs.minute > m - 5)].latency_s.median() for m in range(60)]),     # ventana móvil de 5 min
    "latencia p95 (s)": pd.Series([runs[(runs.minute <= m) & (runs.minute > m - 5)].latency_s.quantile(.95) for m in range(60)]),
    "tokens de entrada / min": g.input_tokens.sum().reindex(range(60), fill_value=0),
    "pasos por petición": g.steps.mean().reindex(range(60)),
})
fig, axes = plt.subplots(2, 3, figsize=(15, 6)); axes = axes.ravel()
cols = ["tráfico (peticiones/min)", "errores (%)", "latencia p50 (s)", "tokens de entrada / min", "pasos por petición"]
for ax, c in zip(axes, cols):
    ax.plot(M.index, M[c], color=viz.PALETTE["traces"], marker="o", ms=2.5, lw=1.4); ax.set_title(c)
axes[2].plot(M.index, M["latencia p95 (s)"], color=viz.PALETTE["error"], lw=1.4, label="p95"); axes[2].plot(M.index, M["latencia p50 (s)"], color=viz.PALETTE["traces"], label="p50"); axes[2].legend(); axes[2].set_title("latencia p50 / p95 (s, ventana móvil de 5 min)")
ax = axes[5]; ax.plot(M.index, runs.groupby("minute").cost_usd.sum().reindex(range(60), fill_value=0) * 1000, color=viz.PALETTE["metrics"]); ax.set_title("coste (milésimas de USD / min)")
for ax in axes:
    ax.axvspan(20, 30, color=viz.PALETTE["error"], alpha=.1); ax.axvspan(40, 48, color=viz.PALETTE["warn"], alpha=.12); ax.set_xlabel("minuto")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - **Franja roja (min 20-30, proveedor LLM degradado):** sube la tasa de **errores**; fíjate que los **tokens de entrada** *bajan* (las llamadas fallidas no consumen tokens) y los **pasos** también: un fallo total *reduce* el coste.
> - **Franja naranja (min 40-48, herramientas lentas):** **no hay errores**, el tráfico y los tokens no cambian… pero la **latencia p95 se duplica o más** mientras la mediana sube mucho menos. Si solo vigilaras errores, no verías este incidente. Hace falta **latencia con percentiles altos** (y ventanas con suficientes peticiones: con pocos datos por minuto el p95 es ruidoso).
> - Ninguno de los seis paneles cuenta si las respuestas fueron **correctas**. Esa es la *quinta* señal que un agente necesita (módulo 14).

---
# 6 · 📐 Percentiles: por qué no se promedian

El percentil **no es lineal**: el p95 de la unión de dos conjuntos **no** es la media de sus p95. Un tablero que «promedia p95 de varias instancias» o «el p95 medio por minuto» da números sin significado estadístico.
Lo correcto es **agregar los histogramas** (sumar los recuentos por bucket) y calcular el percentil sobre el histograma combinado.

$$ p_{95}\big(A\cup B\big)\;\neq\;\tfrac{1}{2}\big(p_{95}(A)+p_{95}(B)\big)\qquad\text{pero}\qquad \text{hist}(A\cup B)=\text{hist}(A)+\text{hist}(B) $$

Tres instancias del agente con tráfico muy desigual (la tercera es lenta pero casi no recibe peticiones):

In [ ]:
rng = np.random.default_rng(SEED)
pods = {"pod-a (90 %)": rng.lognormal(np.log(1.4), 0.30, 9000), "pod-b (9 %)": rng.lognormal(np.log(1.6), 0.35, 900), "pod-c (1 %, lento)": rng.lognormal(np.log(6.0), 0.40, 100)}
allv = np.concatenate(list(pods.values()))
bounds = [0.5, 1, 1.5, 2, 3, 4, 6, 8, 12, 20]
H = {}
for k, v in pods.items():
    h = Histogram(k, bounds); [h.record(x) for x in v]; H[k] = h
merged = Histogram("merged", bounds); merged.counts = [sum(h.counts[i] for h in H.values()) for i in range(len(bounds) + 1)]; merged.count = sum(h.count for h in H.values())
vals = {"promedio de los p95\nde cada pod (MAL)": np.mean([np.quantile(v, .95) for v in pods.values()]), "p95 exacto\nde todas las peticiones": np.quantile(allv, .95),
        "p95 del histograma\nagregado (BIEN)": merged.quantile(.95)}
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(list(vals), list(vals.values()), color=[viz.PALETTE["error"], "k", viz.PALETTE["ok"]], alpha=.85)
for i, v in enumerate(vals.values()): ax.text(i, v + 0.05, f"{v:.2f}s", ha="center")
ax.set_ylabel("latencia p95 (s)"); ax.set_title("El p95 «global»: tres formas de calcularlo"); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - El **promedio de p95** (rojo) está inflado por el pod lento (que solo recibe el 1 % del tráfico pero pesa un tercio en el promedio simple): **sobreestima** mucho el p95 real.
> - El **histograma agregado** (verde) queda cerca del valor exacto (negro); la diferencia es el error de interpolar dentro del bucket (aquí el de 2-3 s).
> - Es la razón por la que Prometheus/OTel prefieren **histogramas** a «p95 precalculado en la aplicación»: los primeros se pueden sumar entre instancias, ventanas y servicios; los segundos, no.

---
# 7 · 🧠 Cardinalidad: lo que rompe un sistema de métricas

En métricas cada **combinación de valores de etiquetas** (*labels*) es una **serie temporal distinta**, y el coste de almacenamiento y consulta crece con el número de series, no con el de eventos. Para un histograma:

$$ \#\text{series} \;=\; \Big(\prod_{l\in\text{labels}} |V_l|\Big)\times(\#\text{buckets}+2) $$

(cada *bucket*, más `_sum` y `_count`, es una serie). Veamos qué ocurre al añadir etiquetas «razonables» a `agent_request_duration_seconds` (10 buckets):

In [ ]:
BUCKETS = 10
labels = [("intent (4)", 4), ("+ status (5)", 5), ("+ modelo (2)", 2), ("+ tier (2)", 2), ("+ canal (3)", 3), ("+ customer_id (200)", 200), ("+ ticket_id (1 por petición: 1000)", 1000)]
series, prod, names = [], 1, []
for n, c in labels:
    prod *= c; series.append(prod * (BUCKETS + 2)); names.append(n)
fig, ax = plt.subplots(figsize=(10, 4))
colors = [viz.PALETTE["ok"] if s < 5_000 else viz.PALETTE["warn"] if s < 100_000 else viz.PALETTE["error"] for s in series]
ax.bar(range(len(series)), series, color=colors); ax.set_yscale("log"); ax.set_xticks(range(len(series))); ax.set_xticklabels(names, rotation=30, ha="right", fontsize=8)
for i, s in enumerate(series): ax.text(i, s * 1.15, f"{s:,}".replace(",", "."), ha="center", fontsize=8)
ax.set_ylabel("series temporales (escala log)"); ax.set_title("Cada etiqueta multiplica las series (histograma de 10 buckets)"); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - Con `intent` + `status` (+ modelo, tier, canal) estás en **cientos o pocos miles** de series: razonable.
> - `customer_id` multiplica **por 200** (varios cientos de miles de series): ya es un problema, y en producción real serán miles de clientes.
> - `ticket_id` (valor único por petición) crea una serie *nueva por petición*: cardinalidad **ilimitada**. Es el error clásico que tumba un Prometheus.
> - **Regla:** los identificadores de alta cardinalidad (ticket, cliente, trace_id) van en **trazas y logs** (donde cada evento ya es único), **no** en etiquetas de métricas. Si necesitas «métricas por cliente», usa pocos grupos (p. ej. `tier`) o atribución a través de trazas/span metrics (módulo 18).

---
# 8 · ¿Qué señal detecta qué fallo? (hipótesis para el proyecto)

Con lo visto en el módulo 01 (huellas de cada fallo) y en esta lección (qué ve cada señal), podemos formular una **hipótesis** de detectabilidad. En el **proyecto** la *medirás* con 10 incidentes simulados; aquí solo la razonamos.
Escala: 2 = la señal lo detecta con claridad · 1 = lo deja ver pero ambiguo/indirecto · 0 = ciega.

In [ ]:
H = pd.DataFrame({
    "métricas (RED + tokens)":  [2, 2, 0, 2, 0, 2, 1, 0],
    "logs":                     [2, 0, 2, 2, 2, 1, 0, 0],
    "trazas":                   [2, 2, 2, 2, 2, 2, 0, 0],
    "contenido / evaluación":   [0, 0, 0, 0, 0, 0, 2, 2],
}, index=["llm_error persistente", "latencia del LLM ×6", "tool_error (degraded)", "tool_timeout", "reintentos 429 (transitorio)", "tool_loop", "alucinación", "fuga de PII"])
fig, ax = plt.subplots(figsize=(8.5, 4.4))
im = ax.imshow(H.values, cmap="RdYlGn", vmin=0, vmax=2, aspect="auto")
ax.set_xticks(range(H.shape[1])); ax.set_xticklabels(H.columns, rotation=20, ha="right"); ax.set_yticks(range(H.shape[0])); ax.set_yticklabels(H.index)
for i in range(H.shape[0]):
    for j in range(H.shape[1]): ax.text(j, i, ["ciega", "indirecta", "clara"][H.values[i, j]], ha="center", va="center", fontsize=8)
ax.set_title("HIPÓTESIS de detectabilidad (la mediremos en el proyecto)"); ax.grid(False); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - Cada fila tiene al menos una señal clara **salvo** alucinaciones y fugas de PII: **ninguno de los tres pilares las ve de forma directa**. Hace falta mirar el **contenido** (evaluación online, filtros de PII): un cuarto «pilar» propio de los agentes.
> - Algunas filas son **ciegas para las métricas** (`tool_error` y los reintentos, porque el agente absorbe el fallo y la petición raíz sale «ok»): ahí mandan logs y trazas. Lección: alertar *solo* sobre la tasa de error de la petición raíz es insuficiente.
> - Esto es una **hipótesis razonada**, no un dato. En el proyecto la contrastarás con mediciones y verás si aguanta.

---
# 9 · 🏭 En producción

- **Google (Dapper, 2010):** el artículo describe la infraestructura de *tracing* distribuido de Google y atribuye parte de su éxito al **muestreo** y a limitar la instrumentación a un pequeño conjunto de librerías comunes. Es el origen conceptual de Zipkin, Jaeger y OpenTelemetry (según resúmenes posteriores) y justifica por qué en producción **no se guarda el 100 % de las trazas** (módulo 18).
- **Honeycomb** (Majors, Fong-Jones y Miranda, *Observability Engineering*, 2022) defiende guardar **eventos estructurados de alta dimensionalidad** y explorar de forma abierta, en lugar de depender solo de métricas pre-agregadas: es la razón de que las trazas con muchos atributos (modelo, tokens, cliente, herramienta) sean tan valiosas en agentes.
- **Grafana Labs** formalizó **RED** (Tom Wilkie, 2015-2018) para servicios y recomienda, junto con la **USE** de Brendan Gregg para recursos, un conjunto **mínimo y consistente** de métricas por servicio: es lo que replicamos para `invoke_agent`, `chat` y `execute_tool`.
- **Google SRE (cap. 6):** las cuatro *golden signals* y la recomendación de **alertar sobre síntomas** (lo que siente el usuario) más que sobre causas internas.
- **OpenTelemetry:** especifica el modelo de datos de las tres señales y de **correlación** entre ellas (trace_id en logs, exemplars en métricas). Los atributos específicos de GenAI (`gen_ai.*`) están en estado *Development*: úsalos con versión fijada (módulo 04).

> **🧠 Secretos de la élite · Secretos de la élite · los tres pilares**
>
> 1. **Cardinalidad = coste (y caídas).** Cada combinación de etiquetas es una serie. Los IDs (cliente, ticket, `trace_id`) pertenecen a trazas y logs; en métricas, pocas dimensiones acotadas (modelo, intención, estado, tier). Controla la cardinalidad con *views* del SDK o el Collector (módulos 05 y 18).
> 2. **Los percentiles no se agregan; los histogramas sí.** Elige los *buckets* alrededor de tus umbrales de SLO. Los histogramas **exponenciales** de OpenTelemetry (si tu backend los soporta) dan buena precisión con pocos parámetros — compruébalo en tu backend antes de adoptarlos.
> 3. **El muestreo sesga las métricas.** Si calculas «tasa de error» solo con las trazas que sobreviven al *tail sampling* (que conserva errores y lentas), tu tasa sale inflada. Las métricas RED deben calcularse **antes** de muestrear (span metrics del Collector, o instrumentos de métricas en la app).
> 4. **El error de la raíz oculta errores de los hijos.** Un agente que absorbe un fallo (reintento, disculpa) deja la petición en `ok`. Alerta sobre **errores por tipo de span** (`execute_tool` por herramienta, `chat` por proveedor), no solo sobre la raíz.
> 5. **Relojes:** las duraciones deben medirse con reloj **monotónico**; las marcas de tiempo de pared (NTP) pueden retroceder y producir spans con duración negativa o hijos «antes» que su padre entre máquinas. Las trazas multi-host siempre tienen algo de *skew*.
> 6. **Eventos de span vs. logs:** un evento dentro del span viaja y se muestrea con la traza (barato, contextual); un log independiente se retiene por separado (útil para auditoría y para lo que ocurre *fuera* de un span). Para un agente: reintentos y excepciones → eventos; auditoría de acciones → logs.
> 7. **Status:** deja `Unset` salvo error; `Ok` (que las librerías *no* deberían establecer) es **final** y suprime errores posteriores en las herramientas de análisis. Ponerlo «por si acaso» esconde fallos.
> 8. **Nombres de span de baja cardinalidad** (`execute_tool get_order`, no `…ORD-00012`): el nombre agrupa en *service maps*, *span metrics* y consultas; el detalle va en atributos.
> 9. **Contenido de prompts y completions = datos sensibles y voluminosos.** No es una métrica ni un log por defecto: se captura de forma opcional, redactada y con retención corta (módulo 20).

> **⚠️ Cuidado · Errores comunes (pitfalls)**
>
> - Promediar percentiles o «p95 medio por instancia».
> - Restar contadores a mano (valores negativos tras un reinicio).
> - Poner `user_id` / `trace_id` como etiqueta de métrica.
> - Alertar solo sobre errores de la petición raíz (los fallos parciales no aparecen).
> - Logs en texto libre sin `trace_id`: no se pueden correlacionar ni consultar por campos.
> - Perder el contexto en colas, hilos o clientes HTTP sin instrumentar: trazas partidas.
> - Medir «tiempo de respuesta» sin separar éxito de error (un error rápido mejora la media).
> - Confundir *métrica de recurso* (CPU del pod) con *saturación del agente* (límite de tasa del proveedor, ventana de contexto).

---
# 10 · Autoevaluación

**Pregunta 1 · (Cálculo) Percentil desde un histograma**

Un histograma con límites `[1, 2, 5]` s tiene recuentos por bucket `(≤1: 50, 1-2: 30, 2-5: 15, >5: 5)`. Estima el p90 por interpolación lineal.

<details>
<summary>Ver respuesta</summary>

N = 100; objetivo = 0,9·100 = 90. Acumulados: 50, 80, 95. El bucket que contiene el rango 90 es (2, 5] con recuento 15 y acumulado previo 80. p90 ≈ 2 + (5−2)·(90−80)/15 = 2 + 3·10/15 = **4,0 s**. Si dijeras «el p90 está entre 2 y 5 s» también sería correcto, pero la interpolación asume reparto uniforme dentro del bucket.

</details>

**Pregunta 2 · (Cálculo) Tasa con reinicio**

Un contador lee, en cuatro lecturas separadas 60 s: 100, 160, 20, 80. ¿Cuántas peticiones por segundo hubo en total en esa ventana (tratando el reinicio)?

<details>
<summary>Ver respuesta</summary>

Incrementos: 100→160 = +60; 160→20 es una **caída** (reinicio): el incremento es el valor actual = 20; 20→80 = +60. Total = 140 en 180 s ⇒ **≈ 0,78 req/s**. La resta ingenua (80−100 = −20) sería absurda.

</details>

**Pregunta 3 · (Cálculo) Cardinalidad**

Quieres un histograma de 12 buckets con etiquetas `intent` (4), `status` (5) y `model` (3). ¿Cuántas series? Si añades `tier` (2) y `customer_id` (300), ¿y ahora?

<details>
<summary>Ver respuesta</summary>

(4·5·3)·(12+2) = 60·14 = **840** series. Con `tier` y `customer_id`: 60·2·300 = 36 000 combinaciones · 14 = **504 000** series (600× más). `customer_id` es el culpable: va a trazas/logs.

</details>

**Pregunta 4 · (Diagnóstico) Trazas partidas**

Tras desplegar un servidor de herramientas, en Tempo ves dos trazas por petición: una con el agente y otra con solo el servidor. ¿Qué ha pasado y cómo lo compruebas?

<details>
<summary>Ver respuesta</summary>

El contexto **no se propaga** entre el agente y el servidor: falta inyectar `traceparent` en la llamada saliente (cliente) o extraerlo en la entrante (servidor), o un proxy/cola descarta la cabecera. Comprobación: capturar la petición saliente y buscar la cabecera `traceparent`; en el servidor, comprobar que el span `SERVER` tiene `parent_span_id` distinto de vacío. Las trazas del servidor tienen un único span raíz con *kind* `SERVER` y sin padre.

</details>

**Pregunta 5 · (Diagnóstico) La alerta que no salta**

Los clientes se quejan de lentitud durante 10 minutos pero tu alerta («tasa de error > 5 %») no se dispara y la latencia *mediana* apenas se mueve. ¿Qué cambias?

<details>
<summary>Ver respuesta</summary>

La latencia degradada afecta a una **fracción** de peticiones (cola): la mediana la oculta y no son errores. Alertar sobre **p95/p99 de latencia** (histograma, ventana adecuada) y, para el mismo síntoma, sobre el **burn rate** de un SLO de latencia (módulo 16). Usa trazas del p99 para localizar el span responsable.

</details>

**Pregunta 6 · (Diagnóstico) «Más barato y rápido»**

El dashboard muestra, tras un cambio de prompt, −40 % de coste y −35 % de latencia sin errores. ¿Qué señal añades para decidir si es bueno?

<details>
<summary>Ver respuesta</summary>

**Pasos/herramientas por petición** (una caída fuerte es sospechosa) y una **señal de calidad** (fundamentación/evaluación online), más una muestra de trazas antes y después. Sin ellas no puedes distinguir eficiencia de regresión silenciosa.

</details>

**Pregunta 7 · (Transferencia) Del ML a los agentes**

En un modelo de recomendación monitorizas CTR (agregado) y revisas casos individuales cuando cae. ¿A qué señal corresponde cada cosa en un agente de soporte?

<details>
<summary>Ver respuesta</summary>

El CTR agregado ≈ **métricas** (tasa de resolución, p95, coste); revisar casos individuales ≈ **trazas** (el recorrido del agente) con **logs** correlacionados y, para la calidad de la respuesta, **evaluación online** (equivalente a las etiquetas tardías).

</details>

**Pregunta 8 · (Transferencia) Una API de pagos**

Aplica *golden signals* a una API de pagos con LLM antifraude. ¿Qué cambia respecto a SupportOps?

<details>
<summary>Ver respuesta</summary>

Latencia (con SLO estricto), tráfico, errores y saturación siguen valiendo; **sube la importancia** de: errores *silenciosos* (falsos negativos del modelo antifraude), de la **auditoría** (logs inmutables con retención y PII redactada) y de la **calidad** medida con etiquetas tardías (contracargos). Cardinalidad: `merchant_id` en trazas, no en métricas.

</details>

---
# 11 · Referencias

- OpenTelemetry, *Specification overview* — <https://github.com/open-telemetry/opentelemetry-specification/blob/main/specification/overview.md>; *Trace API* (SpanContext, SpanKind, Status) — <https://github.com/open-telemetry/opentelemetry-specification/blob/main/specification/trace/api.md>;
  *Logs Data Model* (campos, `SeverityNumber` 1-24) — <https://github.com/open-telemetry/opentelemetry-specification/blob/main/specification/logs/data-model.md>; *Metrics Data Model* (temporalidad, exemplars) — <https://github.com/open-telemetry/opentelemetry-specification/blob/main/specification/metrics/data-model.md>.
- W3C, *Trace Context* (cabecera `traceparent`) — <https://www.w3.org/TR/trace-context/> (la especificación de OpenTelemetry ya referencia también el nivel 2, con el flag *random*).
- Sigelman, Barroso, Burrows et al., *Dapper, a Large-Scale Distributed Systems Tracing Infrastructure*, Google Technical Report dapper-2010-1, 2010 — <https://research.google/pubs/pub36356/>.
- Beyer, Jones, Petoff, Murphy (eds.), *Site Reliability Engineering*, O'Reilly 2016, cap. 6 *Monitoring Distributed Systems* (las cuatro golden signals) — <https://sre.google/sre-book/monitoring-distributed-systems/>.
- Tom Wilkie, *The RED Method: How to Instrument Your Services*, Grafana Labs, 2018 — <https://grafana.com/blog/2018/08/02/the-red-method-how-to-instrument-your-services/>.
- Brendan Gregg, *The USE Method* — <https://www.brendangregg.com/usemethod.html>.
- Charity Majors, Liz Fong-Jones, George Miranda, *Observability Engineering*, O'Reilly, 2022 (ISBN 978-1-492-07644-5).
- Prometheus, *Histograms and summaries* y `histogram_quantile` (documentación oficial) — <https://prometheus.io/docs/practices/histograms/>.
- OpenTelemetry, *GenAI semantic conventions* (estado Development) — <https://github.com/open-telemetry/semantic-conventions-genai>.

**Siguiente:** haz el **proyecto** (`02_proyecto_tres_pilares.ipynb`): clasificar 10 incidentes de SupportOps según qué señal los detecta. Después, módulo 03: producirás estas señales con el SDK de OpenTelemetry.